# Module 9: Year over Year, Rolling Totals and Indexing

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

A monthly report has to say something about the most recent month. There are
three standard ways to do that without being fooled by the calendar, and they
suit different purposes.

This module builds all three, and says which belongs in which document.

**About 15 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
from pathlib import Path

import numpy as np
import pandas as pd

GITHUB = "https://raw.githubusercontent.com/OWNER/REPO/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

print("reading from:", BASE)

monthly = pd.read_csv(BASE + "agency_monthly.csv")
final = monthly[monthly["provisional"] == 0]        # never fit on unfinished months


def series(agency_id, column="n_uof"):
    """One agency's monthly series, indexed by date."""
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    return pd.Series(d[column].values, dtype=float,
                     index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())


def rate(agency_id):
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    return pd.Series((100 * d["n_uof"] / d["n_arrests"]).values,
                     index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())


grandview = series("A012")
print(f"{len(grandview)} months, {grandview.index.min():%Y-%m} to {grandview.index.max():%Y-%m}")

## 2. Year over year

Compare each month to the same month a year earlier. No model, no estimated
factors, and the season cancels because both sides sit at the same point in the
calendar.

In [ ]:
yoy = 100 * (grandview / grandview.shift(12) - 1)

print(yoy.loc["2023"].round(1).to_string())

Cheap and honest, with one cost: it is noisy. Each figure is built from two
single months, so both months' noise goes into it.

In [ ]:
mom = 100 * (grandview / grandview.shift(1) - 1)
print(f"standard deviation of month on month change: {mom.std():.1f} percent")
print(f"standard deviation of year over year change: {yoy.std():.1f} percent")

## 3. Rolling twelve month total

Add up the last twelve months. Every window contains exactly one of each
calendar month, so the season cancels **by construction** rather than by
estimation. No factors, nothing to revise.

In [ ]:
rolling = grandview.rolling(12).sum()

decembers = rolling[rolling.index.month == 12]
print("twelve months ending each December")
print(decembers.astype(int).to_string())

In [ ]:
print(f"variability of the rolling total, as a share of its own mean: "
      f"{100 * rolling.std() / rolling.mean():.1f} percent")
print(f"variability of the raw monthly count:                        "
      f"{100 * grandview.std() / grandview.mean():.1f} percent")

The rolling total is far steadier than the raw series, which is why it is the
usual headline figure in an annual or quarterly report.

Its cost is lag. An event enters the window and stays for twelve months, so the
rolling total responds slowly and then keeps responding long after the event is
over. [Module 8](Module_08_Rolling_Statistics_And_Control_Limits.md) made the
same point about rolling means: good for describing, poor for detecting.

## 4. Indexing to a base period

Divide every value by a base period average and multiply by 100. The base
period becomes 100 and everything else is read as a percentage of it.

This is what puts agencies of wildly different sizes on one chart. An eight
officer department and a 902 officer department cannot share an axis in
incidents. They can share one in "percent of its own 2019".

In [ ]:
def indexed(agency_id, base_year="2019"):
    s = series(agency_id)
    return 100 * s / s.loc[base_year].mean()

table = pd.DataFrame({
    "Ashfell": indexed("A012"),
    "Stonewick": indexed("A001"),
    "Summit County": indexed("A007"),
    "Orrindale": indexed("A006"),
}).rolling(12).mean()      # smoothed, or the small agency is unreadable

table.loc[["2020-12-01", "2022-12-01", "2025-12-01"]].round(0)

Read across a row and you are comparing how far each agency has moved **from
its own starting point**, not how they compare to each other in level. That is
a different question from the one [Module 3](Module_03_Choosing_A_Denominator.md)
asked, and often the more useful one for a trend chart.

Two warnings. The base period must be ordinary, or every later value is measured
against an unusual year. And an index says nothing about level: an agency at 70
may still have a far higher rate than one at 110.

## 5. Which to use

In [ ]:
summary = pd.DataFrame({
    "removes the season": ["yes, exactly", "yes, by construction", "no, keeps it"],
    "needs estimated factors": ["no", "no", "no"],
    "responds quickly": ["yes", "no, lags up to 12 months", "as fast as the input"],
    "noisy": ["yes", "no", "depends on the input"],
    "best for": ["a monthly report line",
                 "an annual headline figure",
                 "comparing agencies of different sizes"],
}, index=["year over year", "rolling 12 month total", "index to a base period"])
summary.T

And a fourth option, from [Module 7](Module_07_Seasonal_Adjustment.md): a
seasonally adjusted series, which is the only one of the four that makes
**consecutive months** comparable. It is also the only one that needs estimated
factors and therefore gets revised.

## Exercise

Build all three presentations for Tarnbridge and find the month where year over
year and the rolling total point in opposite directions.

In [ ]:
# Fill in the blank, then run.
AGENCY = None              # try "A002"

if AGENCY:
    s = series(AGENCY)
    out = pd.DataFrame({
        "count": s,
        "yoy": (100 * (s / s.shift(12) - 1)).round(1),
        "rolling12": s.rolling(12).sum(),
    })
    out["rolling change"] = out["rolling12"].diff().round(0)
    disagree = out[(out["yoy"] > 0) & (out["rolling change"] < 0)].dropna()
    print(f"{len(disagree)} months where year over year is up and the rolling total is down")
    print(disagree.head(6).to_string())
else:
    print("Set AGENCY above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
AGENCY = "A002"
```

There are several such months, and they are not contradictions. Year over year
compares one month to one month. The rolling total compares a twelve month
window to the window one month earlier, which differs only in that it drops one
month and adds another.

So the rolling total can fall while year over year rises whenever the month
that just dropped out of the window was even higher than the month that came
in. Both statements are true about different comparisons, which is exactly why
a report should name the comparison rather than saying "incidents are up".

</details>

---

**Next:** [Module 10, Reading ACF and PACF as Pictures](Module_10_Reading_Autocorrelation.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*